In [4]:
# Step 1: Data Collection & Preprocessing
import pandas as pd
import numpy as np
df = pd.read_csv(r"C:\Users\Admin\Desktop\supplier_risk.csv")
df.head()



,Supplier_ID,Year,Financial_Stability_Score,Delivery_Performance_Score,Quality_Compliance_Score,Regulatory_Adherence_Score,Sustainability_Score,Past_Risk_Level,ERP_Transactions,Incidents_Count,MCDM_Score,Risk_Category
0,52,2025,0.43,0.87,0.80,0.78,0.37,0.46,422,3,0.66,Medium
1,24,2023,0.31,0.98,0.92,0.68,0.43,0.18,363,11,0.67,Medium
2,89,2021,0.67,0.64,0.52,0.99,0.46,0.09,495,18,0.66,Medium
3,55,2024,0.71,0.43,0.80,0.67,0.35,0.95,365,13,0.62,Medium
4,9,2022,0.37,0.81,0.72,0.65,0.65,0.03,255,16,0.62,Medium


In [5]:
df.shape


(1800, 12)

In [6]:
df.columns

Index(['Supplier_ID', 'Year', 'Financial_Stability_Score',
       'Delivery_Performance_Score', 'Quality_Compliance_Score',
       'Regulatory_Adherence_Score', 'Sustainability_Score', 'Past_Risk_Level',
       'ERP_Transactions', 'Incidents_Count', 'MCDM_Score', 'Risk_Category'],
      dtype='object')

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1800 entries, 0 to 1799
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Supplier_ID                 1800 non-null   int64  
 1   Year                        1800 non-null   int64  
 2   Financial_Stability_Score   1800 non-null   float64
 3   Delivery_Performance_Score  1800 non-null   float64
 4   Quality_Compliance_Score    1800 non-null   float64
 5   Regulatory_Adherence_Score  1800 non-null   float64
 6   Sustainability_Score        1800 non-null   float64
 7   Past_Risk_Level             1800 non-null   float64
 8   ERP_Transactions            1800 non-null   int64  
 9   Incidents_Count             1800 non-null   int64  
 10  MCDM_Score                  1800 non-null   float64
 11  Risk_Category               1800 non-null   object 
dtypes: float64(7), int64(4), object(1)
memory usage: 168.9+ KB


In [8]:
df.describe()

,Supplier_ID,Year,Financial_Stability_Score,Delivery_Performance_Score,Quality_Compliance_Score,Regulatory_Adherence_Score,Sustainability_Score,Past_Risk_Level,ERP_Transactions,Incidents_Count,MCDM_Score
count,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000
mean,49.173889,2022.958333,0.647500,0.699083,0.746772,0.796194,0.646228,0.494578,270.117222,9.601111,0.702533
std,28.191087,1.420464,0.199539,0.174938,0.140464,0.115470,0.202019,0.291539,130.734123,5.758605,0.085498
min,1.000000,2021.000000,0.300000,0.400000,0.500000,0.600000,0.300000,0.000000,50.000000,0.000000,0.430000
25%,25.000000,2022.000000,0.470000,0.540000,0.630000,0.700000,0.470000,0.240000,158.000000,5.000000,0.640000
50%,49.000000,2023.000000,0.650000,0.700000,0.740000,0.800000,0.640000,0.500000,264.000000,10.000000,0.700000
75%,73.000000,2024.000000,0.810000,0.850000,0.870000,0.890000,0.820000,0.740000,385.000000,15.000000,0.760000
max,100.000000,2025.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,499.000000,19.000000,0.930000


In [9]:
#PREPROCESSING USING PYSPARK DATAFRAME API

from pyspark.sql import SparkSession
from pyspark.sql.functions import when
spark = SparkSession.builder.appName("SupplierRiskProject").getOrCreate()
supplier_risk_df = spark.read.csv("supplier_risk.csv", header=True, inferSchema=True)


ModuleNotFoundError: No module named 'pyspark'

In [10]:
print("Before conversion:")
supplier_risk_df.select("Risk_Category").show(10)
supplier_risk_df.printSchema()

Before conversion:


NameError: name 'supplier_risk_df' is not defined

In [ ]:
# Show number of rows and columns
print("Number of rows:", supplier_risk_df.count())
print("Number of columns:", len(supplier_risk_df.columns))
print("Column names:", supplier_risk_df.columns)

In [ ]:
# Check missing values
for column in supplier_risk_df.columns:
    missing_count = supplier_risk_df.filter(supplier_risk_df[column].isNull()).count()
    print(f"{column}: {missing_count} missing values")

In [ ]:
# Remove duplicates
before_rows = supplier_risk_df.count()
supplier_risk_df = supplier_risk_df.dropDuplicates()
after_rows = supplier_risk_df.count()
print("Rows before duplicate removal:", before_rows)
print("Rows after duplicate removal:", after_rows)

In [ ]:
# Drop ID column
supplier_risk_df = supplier_risk_df.drop("Supplier_ID")

In [ ]:
# Convert target labels into numeric values
supplier_risk_df = supplier_risk_df.withColumn(
    "Risk_Category",
    when(supplier_risk_df["Risk_Category"] == "Medium", 0)
    .when(supplier_risk_df["Risk_Category"] == "High", 1)
    .otherwise(None)
)

In [ ]:
print("After conversion:")
supplier_risk_df.select("Risk_Category").show(10)
supplier_risk_df.printSchema()

In [ ]:
#  preview
supplier_risk_df.show(10)
supplier_risk_df.printSchema()

In [ ]:
#check & handle outlier
df = supplier_risk_df.toPandas()
df.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.boxplot(data=df[["Financial_Stability_Score", "Delivery_Performance_Score", "MCDM_Score"]])
plt.title("Boxplot for Selected Numerical Features")
plt.show()

In [ ]:
import numpy as np
numerical_cols = df.select_dtypes(include=np.number).columns.drop("Risk_Category")
print(numerical_cols)

In [ ]:
for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: {outliers} outliers after capping")

In [ ]:
#### 2 Exploratory Data Analysis (EDA)
###2.2 correlations between variables

import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
### 2.3 Check class imbalance in supplier reliability
df["Risk_Label"] = df["Risk_Category"].map({0: "Medium", 1: "High"})

sns.countplot(x="Risk_Label", data=df)
plt.title("Distribution of Supplier Risk Category")
plt.xlabel("Risk Category")
plt.ylabel("Count")
plt.show()

print(df["Risk_Label"].value_counts())
print(df["Risk_Label"].value_counts(normalize=True))

In [ ]:
### 3. Feature Engineering

In [ ]:
###3.1 Create new features
# Feature 1: Incident Rate
df["Incident_Rate"] = df["Incidents_Count"] / (df["ERP_Transactions"] + 1)

# Preview
df[["Incidents_Count", "ERP_Transactions", "Incident_Rate"]].head()

Reason for creating this feature:
Incidents_Count alone does not fully reflect supplier risk, because suppliers may operate at very different transaction volumes. A supplier with more incidents may also process many more transactions. Therefore, Incident_Rate provides a normalized measure of operational issues by dividing the number of incidents by the number of ERP transactions.

Example:
For example, Supplier A may have 10 incidents with 5,000 transactions, while Supplier B has 5 incidents with only 100 transactions. If we only look at the total number of incidents, Supplier A appears riskier. However, when we calculate Incident_Rate, Supplier B may actually have a much higher rate of operational problems relative to its transaction volume.

Why it is useful:
This feature helps compare suppliers more fairly. Instead of looking only at the total number of incidents, we consider incidents relative to activity volume. This can improve the model’s ability to identify risky suppliers more accurately.

In [ ]:
# Feature 2: Resilience Score
df["Resilience_Score"] = (
    df["Financial_Stability_Score"] +
    df["Sustainability_Score"] -
    df["Past_Risk_Level"]
)

# Preview
df[["Financial_Stability_Score", "Sustainability_Score", "Past_Risk_Level", "Resilience_Score"]].head()

Reason for creating this feature:
A supplier’s resilience can be represented by combining positive indicators and negative historical risk. In this project, Resilience_Score is created by adding Financial_Stability_Score and Sustainability_Score, then subtracting Past_Risk_Level.

Why it is useful:
This feature provides a simple summary of supplier strength and long-term stability. Financial stability and sustainability are positive attributes, while past risk level is a negative factor. Combining them into one score may help the model capture supplier resilience more effectively than using separate variables alone.

In [ ]:
# Final check of new features
df[[
    "Incidents_Count", "ERP_Transactions", "Incident_Rate",
    "Financial_Stability_Score", "Sustainability_Score",
    "Past_Risk_Level", "Resilience_Score"
]].head()

In [ ]:
##3.2 Scale numerical variables
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=["Risk_Category", "Risk_Label", "MCDM_Score"])
y = df["Risk_Category"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

#3.3 Encode categorical variables

No additional categorical encoding was required because the Risk_Category column had already been converted from text labels (Medium, High) to numeric values (0, 1) in Step 1 during data preprocessing.

In [ ]:
#Baseline models(Random Forest and Logistic Regression)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


log_reg = LogisticRegression(max_iter=500)
log_reg.fit(X_train_scaled, y_train)
log_pred = log_reg.predict(X_test_scaled)

print("Logistic Regression Baseline")
print(classification_report(y_test, log_pred))



rf_base = RandomForestClassifier(random_state=42)
rf_base.fit(X_train_scaled, y_train)
rf_pred = rf_base.predict(X_test_scaled)

print("Random Forest Baseline")
print(classification_report(y_test, rf_pred))

In [ ]:
print(df.duplicated().sum())
print(X.columns)
print(y_test.value_counts())

In [ ]:
#Tuning hyperparameters for Random Forest
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

rf = RandomForestClassifier(random_state=42)

param_dist = {
    'n_estimators': randint(100, 500),
    'max_depth': randint(3, 30),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'bootstrap': [True, False],
    'max_features': ['sqrt', 'log2', None]
}

rand_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=30,
    scoring='f1',
    cv=5,
    random_state=42,
    n_jobs=-1
)

rand_search.fit(X_train_scaled, y_train)

best_rf = rand_search.best_estimator_

print("Best Hyperparameters Found")
print(rand_search.best_params_)

In [ ]:
# Final tuned RF predictions
best_rf_pred = best_rf.predict(X_test_scaled)

print("FINAL Tuned Random Forest Model")
print(classification_report(y_test, best_rf_pred))


In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, best_rf_pred)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, cmap='Blues', fmt='d')
plt.title("Confusion Matrix - Tuned Random Forest")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, log_pred)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, cmap='Blues', fmt='d')
plt.title("Confusion Matrix - Tuned LR")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt


# Logistic Regression: probability of class 1
log_prob = log_reg.predict_proba(X_test_scaled)[:, 1]

# Random Forest: probability of class 1
rf_prob = best_rf.predict_proba(X_test_scaled)[:, 1]

log_fpr, log_tpr, _ = roc_curve(y_test, log_prob)
log_auc = roc_auc_score(y_test, log_prob)

rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_prob)
rf_auc = roc_auc_score(y_test, rf_prob)

plt.figure(figsize=(8,6))
plt.plot(log_fpr, log_tpr, label=f"Logistic Regression (AUC = {log_auc:.2f})", linewidth=2)
plt.plot(rf_fpr, rf_tpr, label=f"Random Forest (AUC = {rf_auc:.2f})", linewidth=2)

# Diagonal baseline
plt.plot([0,1], [0,1], 'k--', label="Random Guess")

plt.title("ROC Curve Comparison")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np

importances = best_rf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(9,6))
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)), X_train.columns[indices], rotation=90)
plt.title("Feature Importance - Tuned Random Forest")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Get coefficients
coefficients = log_regcoefficients = log_reg.coef_[0]
importance = np.abs(coefficients)

# Map to feature names
feature_names = X_train.columns

# Sort for better visualization
indices = np.argsort(importance)[::-1]
sorted_importance = importance[indices]
sorted_features = feature_names[indices]

# Plot
plt.figure(figsize=(10,6))
plt.bar(range(len(sorted_importance)), sorted_importance)
plt.xticks(range(len(sorted_importance)), sorted_features, rotation=90)
plt.title("Feature Importance (Coefficient Magnitudes) - Logistic Regression")
plt.ylabel("Coefficient Magnitude")
plt.show()


Insight 1: Financial Stability is the Primary Risk Driver
Interpretation:
Suppliers with weak financial stability are far more likely to become high‑risk.

Actions for managers:
Implement early‑warning financial dashboards tracking liquidity, solvency, and credit scores.
Prioritize financial audits of suppliers with declining financial metrics.
Require more frequent financial disclosures for critical suppliers.





Insight 2: Delivery Performance Strongly Predicts Operational Reliability

Interpretation:
Late deliveries or inconsistent delivery performance are major indicators of future disruptions.

Actions:

Use models to flag suppliers trending downward in on‑time delivery and shift volume away from suppliers showing declining performance before service failures occur.
Introduce performance‑based contracts or penalties for chronic delivery issues.

Insight 3: Supplier Resiliency Matters — Diversify Risk Early

Interpretation:
Resiliency Score (financial + sustainability – historical risk) shows structural strength.

Actions:

Take resiliency scoring into account when making contracts.
Provide development programs/training for low‑resiliency suppliers.

This analysis highlights that supplier risk is driven primarily by financial stability and delivery performance. Supply chain managers should focus monitoring resources and intervention programs on suppliers exhibiting weaknesses in these dimensions. Resiliency and quality metrics also provide valuable insights and should inform sourcing decisions.
Overall, the model provides evidence‑based prioritisation that can improve supplier evaluations, sourcing strategies, and disruption prevention planning.

